# Phase 2: controls, Dangerzone answers and the adaptive pilot

Use **Runtime > Run all**. The cells run in order and the long one (cell 5) stays in the
foreground, so Colab sees an active cell. Runtime type must be a GPU (T4 or better).

| Cell | What it does |
|---|---|
| 1 | Settings (stages, time limit, auto-disconnect) |
| 2 | Mounts Drive, checks the GPU, checks the inputs on Drive |
| 3 | Clones or updates `master` from GitHub |
| 4 | Starts a background copy of `results/runs` to Drive every 5 minutes |
| 5 | Runs the stages and prints progress every minute |
| 6 | Final copy to Drive, summary, and disconnect when everything succeeded |

Stages, in order: `controls` (13 defenses on the stored `full_200` adversarial images),
`dz_answers` (answers on the locally sanitized Dangerzone images), `pilot` (adaptive attack,
10 images, 16/255, through JPEG 50, Dangerzone-lossless and legacy Dangerzone),
`pilot_eval` (defended-loss check, gate G3).

Every job is idempotent. If the credits run out or the session is reclaimed, open a new
session and Run all again: finished jobs are skipped and at most the last 5 minutes of
work is lost. LLaVA downloads on each new session (about 11 minutes).

`pilot_dz` is left out on purpose. It needs the pilot adversarial images sanitized by real
Dangerzone on a rootless-Podman machine first; run it afterwards with `STAGES = "pilot_dz"`.

**Later stages** (set `STAGES` in cell 1; protocol in `docs/phase2_screen_plan.md`):
`jpeg_ext` (staged adaptive JPEG 50 attack, 500/1000/2000 steps, stops by a fixed rule),
`screen` (50-image adaptive screen, `SCREEN_STEPS` per attack, `SCREEN_ONLY` picks runs),
`screen_eval` (defended-loss gate), `screen_dz` (answers on locally sanitized Dangerzone images).


In [ ]:
# 1. Settings
STAGES = "screen"
SESSION_HOURS = 11.0        # limit passed to each run_experiment call; the Colab credit is the real limit
AUTO_DISCONNECT = False     # release the runtime after a fully successful run (saves credit)
BRANCH = "master"
REPO_URL = "https://github.com/nongvantinh/computer-vision.git"
DRIVE_DIR = "/content/drive/MyDrive/computer-vision"
BASE_RUN = "full_200"
SCREEN_STEPS = 500          # step budget of every adaptive attack in the screen
SCREEN_ONLY = "screen_dz_ll_e16"   # comma list of screen run ids to run, empty means all

In [ ]:
# 2. Drive, GPU and input checks (fails here, before the 11 minute model download)
import os, subprocess, time
from google.colab import drive
drive.mount("/content/drive")

LOCAL, REMOTE = "/content/cdr/results/runs", f"{DRIVE_DIR}/results/runs"
KEEP = ("--include='*/' --include='*.json' --include='*.jsonl' --include='*.csv' "
        "--include='*.yaml' --include='*.png' --include='*.npy' --exclude='*'")

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

gpu = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
assert gpu, "No GPU. Use Runtime > Change runtime type > T4 GPU, then Run all again."
print("GPU:", gpu)

stages = [s.strip() for s in STAGES.split(",") if s.strip()]
need_base = any(s in stages for s in ("controls", "dz_answers", "pilot_eval"))
if need_base:
    n_adv = int(sh(f"find {REMOTE}/{BASE_RUN}/adv -name '*.npy' 2>/dev/null | wc -l") or 0)
    print(f"{BASE_RUN}: {n_adv} stored adversarial images on Drive")
    assert n_adv > 0, f"{REMOTE}/{BASE_RUN}/adv is missing or empty"
if "dz_answers" in stages:
    for d in ("dz", "dz_ll"):
        n = int(sh(f"ls {REMOTE}/{BASE_RUN}/{d} 2>/dev/null | wc -l") or 0)
        print(f"{BASE_RUN}/{d}: {n} files" + ("" if n else "   <-- missing, dz_answers will skip it"))
for r in ("controls_v1", "pilot_adaptive_jpeg50", "pilot_adaptive_dz_ll", "pilot_adaptive_dz"):
    n = sh(f"find {REMOTE}/{r} -name '*.json' 2>/dev/null | wc -l")
    print(f"{r}: {n} json files already on Drive")

In [ ]:
# 3. Code from GitHub
if not os.path.exists("/content/cdr/.git"):
    subprocess.run(f"git clone -q --branch {BRANCH} {REPO_URL} /content/cdr", shell=True, check=True)
subprocess.run(f"git -C /content/cdr fetch -q --all && git -C /content/cdr checkout -q {BRANCH} "
               f"&& git -C /content/cdr pull -q --ff-only", shell=True, check=True)
print(sh("git -C /content/cdr log -1 --oneline"))

In [ ]:
# 4. Copy to Drive. The stage script copies every run directory to Drive every 2 minutes
# and prints 'SYNC <run>: adv local N, on Drive M'. Keep this browser tab open: Colab
# releases the runtime a while after the last tab closes.
sync = type('S', (), {'terminate': lambda self: None})()
print('sync is handled inside the stage script')

In [ ]:
# 5. Run the stages. This cell stays busy until they finish.
open("/content/run_stage.py", "w").write(
    f'STAGE="{STAGES}"\nBRANCH="{BRANCH}"\nSESSION_HOURS={SESSION_HOURS}\n'
    f'DRIVE_DIR="{DRIVE_DIR}"\nBASE_RUN="{BASE_RUN}"\n'
    f'SCREEN_STEPS={SCREEN_STEPS}\nSCREEN_ONLY="{SCREEN_ONLY}"\n'
    f'exec(open("/content/cdr/scripts/colab_phase2.py").read())\n')
LOG = "/content/phase2.log"
t0 = time.time()
proc = subprocess.Popen(f"cd /content/cdr && python /content/run_stage.py > {LOG} 2>&1", shell=True)
pos = 0
while proc.poll() is None:
    time.sleep(60)
    with open(LOG, "rb") as f:
        f.seek(pos); chunk = f.read(); pos = f.tell()
    lines = [l for l in chunk.decode(errors="replace").replace("\r", "\n").splitlines()
             if l.strip() and "Loading weights" not in l]
    if lines:
        print(f"[{(time.time() - t0) / 60:6.1f} min]", flush=True)
        print("\n".join(lines[-8:]), flush=True)
RC = proc.returncode
print("exit code:", RC, f"after {(time.time() - t0) / 60:.1f} min")

In [ ]:
# 6. Final copy to Drive, summary, optional disconnect
for d in sorted(os.listdir(LOCAL)):
    sh(f"mkdir -p {REMOTE}/{d} && rsync -a {KEEP} {LOCAL}/{d}/ {REMOTE}/{d}/")
sh(f"cp {LOG} {DRIVE_DIR}/phase2_last.log")
sync.terminate()
log = open(LOG, errors="replace").read()
done = [l for l in log.splitlines() if l.startswith("STAGE DONE")]
print("\n".join(done) or "no stage finished")
print("\n--- last lines of the log ---")
print("\n".join(log.splitlines()[-6:]))
ok = RC == 0 and len(done) == len(stages)
print("\nALL STAGES FINISHED" if ok else "\nNOT FINISHED: re-run all cells to resume (the runtime was kept).")
if ok and AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()